# Train Baseline LSTM on Google Colab

This notebook runs the existing baseline LSTM training script. It does not recreate the dataset, modify preprocessing outputs, or duplicate the training loop.

Before running training, select a GPU runtime: `Runtime > Change runtime type > GPU`.

## 1. Mount Google Drive

In [14]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Set Project Paths

This assumes the project root is `/content/drive/MyDrive/Seoul_bike_project`. The dataset is copied from Drive to local Colab disk before training.

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path("/content/drive/MyDrive/Seoul_bike_project")
DRIVE_DATA_DIR = PROJECT_DIR / "data" / "lstm_baseline"
LOCAL_DATA_DIR = Path("/content/lstm_baseline")
DATA_DIR = LOCAL_DATA_DIR
CONFIG_PATH = PROJECT_DIR / "configs" / "lstm_baseline.yaml"
BATCH_CONFIG_PATH = PROJECT_DIR / "configs" / "lstm_baseline_batch_32768.yaml"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints" / "lstm_baseline"
MODEL_DIR = PROJECT_DIR / "models" / "lstm_baseline"
LOG_DIR = PROJECT_DIR / "logs" / "lstm_baseline"

os.chdir(PROJECT_DIR)
print("Project directory:", PROJECT_DIR)
print("Current working directory:", Path.cwd())
print("Drive data directory:", DRIVE_DATA_DIR)
print("Local Colab data directory:", DATA_DIR)

## 3. Install Dependencies

This keeps the setup simple for Colab. If packages are already installed, pip will skip or reuse them.

In [16]:
!pip install -q torch numpy pyyaml tqdm wandb

## 4. Verify GPU

## 5. Copy Dataset to Colab Disk

In [ ]:
from pathlib import Path
import shutil

PROJECT_DIR = Path("/content/drive/MyDrive/Seoul_bike_project")
DRIVE_DATA_DIR = PROJECT_DIR / "data" / "lstm_baseline"
LOCAL_DATA_DIR = Path("/content/lstm_baseline")
DATA_DIR = LOCAL_DATA_DIR
RELOAD_LOCAL_DATA = True

if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f"Drive dataset directory does not exist: {DRIVE_DATA_DIR}")

if LOCAL_DATA_DIR.exists() and RELOAD_LOCAL_DATA:
    shutil.rmtree(LOCAL_DATA_DIR)

if not LOCAL_DATA_DIR.exists():
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    print(f"Copied dataset from {DRIVE_DATA_DIR} to {LOCAL_DATA_DIR}")
else:
    print(f"Using existing local dataset at {LOCAL_DATA_DIR}")

print("Training/evaluation data directory:", DATA_DIR)

In [17]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available. Go to Runtime > Change runtime type > GPU.")

## 6. Verify Required Files

In [18]:
required_files = [
    CONFIG_PATH,
    BATCH_CONFIG_PATH,
    PROJECT_DIR / "src" / "training" / "lstm_training" / "train_lstm.py",
    DATA_DIR / "dynamic_features.npy",
    DATA_DIR / "targets.npy",
    DATA_DIR / "targets_raw.npy",
    DATA_DIR / "static_numeric.npy",
    DATA_DIR / "splits.json",
    DATA_DIR / "scalers.json",
    DATA_DIR / "feature_config.json",
    DATA_DIR / "dataset_summary.json",
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("All required files found.")

## 7. Optional: Inspect Dataset Summary

In [19]:
import json

summary_path = DATA_DIR / "dataset_summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

summary

## 8. W&B Login

Do not paste API keys into this notebook. Put your W&B API key in Colab Secrets as `WANDB_API_KEY`, or authenticate separately with `wandb.login()`.

Do not commit API keys to GitHub.

In [20]:
import os
import wandb

try:
    from google.colab import userdata
    wandb_key = userdata.get("WANDB_API_KEY")
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
except Exception:
    pass

wandb.login()

## 9. Fresh Training

Runs a new training attempt using the existing dataset and config.

In [ ]:
!python src/training/lstm_training/train_lstm.py \
  --config configs/lstm_baseline_batch_32768.yaml \
  --data_dir /content/lstm_baseline \
  --checkpoint_dir /content/drive/MyDrive/Seoul_bike_project/checkpoints/lstm_baseline_batch_32768 \
  --model_dir /content/drive/MyDrive/Seoul_bike_project/models/lstm_baseline_batch_32768 \
  --log_dir /content/drive/MyDrive/Seoul_bike_project/logs/lstm_baseline_batch_32768 \
  --wandb_enabled true

## 10. Resume Training After Colab Disconnect

Uses `last.pt` automatically when it exists. This is the recommended crash-recovery path.

In [ ]:
!python src/training/lstm_training/train_lstm.py \
  --config configs/lstm_baseline.yaml \
  --data_dir /content/lstm_baseline \
  --checkpoint_dir /content/drive/MyDrive/Seoul_bike_project/checkpoints/lstm_baseline \
  --model_dir /content/drive/MyDrive/Seoul_bike_project/models/lstm_baseline \
  --log_dir /content/drive/MyDrive/Seoul_bike_project/logs/lstm_baseline \
  --resume auto \
  --resume_mode full \
  --wandb_enabled true

## 11. Resume From Best Checkpoint With Changed Hyperparameters

`weights_only` loads the model weights and reinitializes optimizer/scheduler state from the current command/config.

In [ ]:
!python src/training/lstm_training/train_lstm.py \
  --config configs/lstm_baseline.yaml \
  --data_dir /content/lstm_baseline \
  --checkpoint_dir /content/drive/MyDrive/Seoul_bike_project/checkpoints/lstm_baseline \
  --model_dir /content/drive/MyDrive/Seoul_bike_project/models/lstm_baseline \
  --log_dir /content/drive/MyDrive/Seoul_bike_project/logs/lstm_baseline \
  --resume /content/drive/MyDrive/Seoul_bike_project/checkpoints/lstm_baseline/best.pt \
  --resume_mode weights_only \
  --learning_rate 0.0005 \
  --batch_size 4096 \
  --max_epochs 12 \
  --wandb_enabled true

## 12. Run Without W&B

Use this if W&B is not configured or you want a local-only run.

In [ ]:
!python src/training/lstm_training/train_lstm.py \
  --config configs/lstm_baseline.yaml \
  --data_dir /content/lstm_baseline \
  --checkpoint_dir /content/drive/MyDrive/Seoul_bike_project/checkpoints/lstm_baseline \
  --model_dir /content/drive/MyDrive/Seoul_bike_project/models/lstm_baseline \
  --log_dir /content/drive/MyDrive/Seoul_bike_project/logs/lstm_baseline \
  --wandb_enabled false

## 13. Inspect Final Metrics

In [21]:
import json
from pathlib import Path

metrics_path = LOG_DIR / "final_metrics.json"
if metrics_path.exists():
    with open(metrics_path, "r", encoding="utf-8") as f:
        metrics = json.load(f)
    metrics
else:
    print("No final_metrics.json found yet.")